# 03. Walk-forward 확률 보정

실행 결과는 `results/`에 저장됩니다.

In [ ]:
from pathlib import Path
import sys

experiment_dir = Path.cwd() / "0826" if (Path.cwd() / "0826").exists() else Path.cwd()
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))


In [ ]:
"""CatBoost OOF 확률을 과거 Fold만 이용해 평균 방향으로 보정한다."""

import numpy as np

from common import RESULTS_DIR, brier_metrics, print_metrics, save_json


def fit_shrinkage(y: np.ndarray, pred: np.ndarray) -> tuple[float, float]:
    """p' = rate + alpha * (p - rate)의 Brier 최적 alpha를 구한다."""
    rate = float(y.mean())
    centered = pred - rate
    denominator = float(np.dot(centered, centered))
    if denominator == 0:
        return 0.0, rate
    alpha = float(np.dot(centered, y - rate) / denominator)
    return float(np.clip(alpha, 0.0, 2.0)), rate


def apply_shrinkage(pred: np.ndarray, alpha: float, rate: float) -> np.ndarray:
    return np.clip(rate + alpha * (pred - rate), 0.0, 1.0)


def main() -> None:
    path = RESULTS_DIR / "02_catboost_oof.npz"
    if not path.exists():
        raise FileNotFoundError("먼저 02_catboost_time_cv.py를 실행하세요.")
    data = np.load(path)
    y, pred, year = data["y"], data["prediction"], data["year"]
    rows = []

    # 2022 예측으로 2023을, 2022~2023 예측으로 2024를 보정한다.
    for valid_year in (2023, 2024):
        calibration_mask = year < valid_year
        valid_mask = year == valid_year
        alpha, rate = fit_shrinkage(y[calibration_mask], pred[calibration_mask])
        calibrated = apply_shrinkage(pred[valid_mask], alpha, rate)
        raw_metrics = brier_metrics(y[valid_mask], pred[valid_mask])
        calibrated_metrics = brier_metrics(y[valid_mask], calibrated)
        print_metrics(f"{valid_year} 원본", raw_metrics)
        print_metrics(f"{valid_year} 보정", calibrated_metrics)
        rows.append(
            {
                "valid_year": valid_year,
                "alpha": alpha,
                "calibration_rate": rate,
                "raw": raw_metrics,
                "calibrated": calibrated_metrics,
            }
        )

    final_alpha, final_rate = fit_shrinkage(y, pred)
    payload = {
        "method": "mean shrinkage",
        "formula": "rate + alpha * (prediction - rate)",
        "walk_forward_results": rows,
        "final_parameters_for_retrain": {
            "alpha": final_alpha,
            "rate": final_rate,
        },
    }
    save_json(RESULTS_DIR / "03_calibration_metrics.json", payload)
    print(f"최종 재학습용 alpha={final_alpha:.6f}, rate={final_rate:.6f}")


if __name__ == "__main__":
    main()

